In [1]:
import torch
from torch_geometric.data import Data

/Users/tungvuduc/opt/anaconda3/envs/dlbio_arm64/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# let's do a small example
edge_index = torch.tensor([[0, 1],
                           [1, 0], 
                           [1, 2],
                           [2, 1]], dtype=torch.long)

x = torch.tensor([[-1], [0], [1]], dtype=torch.float)
data = Data(x=x, edge_index=edge_index.T)

print(data)

Data(x=[3, 1], edge_index=[2, 4])


In [3]:
print(data.num_nodes) # print number of nodes

print(data.num_edges) # print number of edges

print(data.num_node_features) # print dim of node features

print(data.has_self_loops()) # print if nodes have connections with themselves

print(data.has_isolated_nodes()) # does the data have isolated nodes?

print(data.is_directed()) # is the data directed?

3
4
1
False
False
False


In [4]:
# let's download some data from torch_geometrics
from torch_geometric.datasets import TUDataset

dataset = TUDataset(root="/tmp/ENZYMES", name="ENZYMES")
print("Dataset consists of", len(dataset), "graphs")
graph = dataset[0] # let's index the first graph
print(graph) # retrieve the first graph

print(graph.num_nodes)
print(graph.num_edges)
print(graph.num_node_features)

print(graph.is_undirected()) # is the graph undirected?
print(graph.has_self_loops()) # does the graph have nodes with connections with themselves?
print(graph.has_isolated_nodes()) # does the graph have isolated nodes?

# let's split the data in a train and test dataset
train_size = 0.8
train_split = round(len(dataset) * 0.8)

dataset = dataset.shuffle()
train_data = dataset[:train_size][0] # we can simply index the dataset object
test_data = dataset[train_size:][0]


Dataset consists of 600 graphs
Data(edge_index=[2, 168], x=[37, 3], y=[1])
37
168
3
True
False
False


In [5]:
# let's download another dataset - this is a citation dataset
from torch_geometric.datasets import Planetoid

dataset = Planetoid(root="/tmp/Cora", name="Cora")
graph = dataset[0]

# let's inspect the dataset
print(graph)
print(f"The dataset consists of {len(graph)} graphs")
print(f"The dataset consists of {graph.num_nodes} nodes")
print(f"The dataset consists of {graph.num_edges} edges")
print(f"The nodes in the dataset consists of {graph.num_node_features} features")
print(f"Is the data directed: {graph.is_directed()}")

print(f"The graph's train size {graph.train_mask.sum().item()}")
print(f"The graph's val size {graph.val_mask.sum().item()}")
print(f"The graph's test size {graph.test_mask.sum().item()}")

Data(x=[2708, 1433], edge_index=[2, 10556], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708])
The dataset consists of 6 graphs
The dataset consists of 2708 nodes
The dataset consists of 10556 edges
The nodes in the dataset consists of 1433 features
Is the data directed: False
The graph's train size 140
The graph's val size 500
The graph's test size 1000


In [35]:
from torch_geometric.datasets import TUDataset
from torch_geometric.loader import DataLoader

dataset = TUDataset(root="/tmp/ENZYMES", name="ENZYMES", use_node_attr=True).shuffle()
loader = DataLoader(dataset=dataset, batch_size=32, shuffle=True)
first_batch = [*loader][0]
print(first_batch.num_graphs) # this is the batch_size

from torch_geometric.utils import scatter
x = scatter(src=first_batch.x, index=first_batch.batch, dim=0, reduce="mean")
print(x.shape)

32
torch.Size([32, 21])
